In [1]:
import requests
import pandas as pd
import numpy as np
import time
from google.colab import drive

In [2]:

drive.mount('/content/drive')


API_KEY = '75ca7dac41122102527fad5e569f5b07928d9225'

DRIVE_FOLDER = '/content/drive/My Drive/MarketplaceApps/CensusProjectFiles/'

Mounted at /content/drive


In [3]:
# Add a prefix so your first and second runs don't overwrite each other!
FILE_PREFIX = 'income-housing-health_'

# 3. The Variable Mapping Dictionary
# Format: 'API_CODE': 'Human_Readable_Header'

VARIABLE_MAP = {
    # --- ECONOMIC CHARACTERISTICS ---
    'B19013_001E': 'MedianHouseholdIncome',
    'B19301_001E': 'PerCapitaIncome',
    'B19083_001E': 'GiniIndex',
    'B19001_001E': 'HHIncome_Total',
    'B19001_002E': 'HHIncome_Under10K',
    'B19001_003E': 'HHIncome_10Kto15K',
    'B19001_011E': 'HHIncome_50Kto60K',
    'B19001_014E': 'HHIncome_100Kto125K',
    'B19001_016E': 'HHIncome_150Kto200K',
    'B19001_017E': 'HHIncome_200KPlus',
    'B17001_001E': 'PovertyStatus_Total',
    'B17001_002E': 'IncomeBelowPoverty',
    'B08301_001E': 'MeansOfTransport_Total',
    'B08301_010E': 'PublicTransit',
    'B08301_021E': 'WorkedFromHome',
    'B08303_001E': 'TravelTimeToWork_Total',
    'B24080_001E': 'ClassOfWorker_Total',
    'B24080_002E': 'PrivateProfitWageSalWorker',
    'B24080_009E': 'LocalGovWorker',
    'B24050_004E': 'SelfEmpIncorporated',
    'B24050_005E': 'SelfEmpUnincorporated',

    # --- HOUSING AFFORDABILITY ---
    'B25071_001E': 'MedianRentAsPctOfIncome',
    'B25014_001E': 'OccupantsPerRoom_Total',

    # --- DIGITAL ACCESS ---
    'B28002_001E': 'InternetAccess_TotalHH',
    'B28002_002E': 'WithInternetSub',
    'B28002_013E': 'NoInternetAccess',

    # --- HEALTH INSURANCE ---
    'B27001_001E': 'HealthIns_Total',
    'B27001_005E': 'NoHealthIns_Male_18to25'

}

# Extract just the API codes to send to the Census
TARGET_VARIABLES = list(VARIABLE_MAP.keys())
print(TARGET_VARIABLES)

years_acs1 = list(range(2008, 2025))
years_acs5 = list(range(2009, 2025))

['B19013_001E', 'B19301_001E', 'B19083_001E', 'B19001_001E', 'B19001_002E', 'B19001_003E', 'B19001_011E', 'B19001_014E', 'B19001_016E', 'B19001_017E', 'B17001_001E', 'B17001_002E', 'B08301_001E', 'B08301_010E', 'B08301_021E', 'B08303_001E', 'B24080_001E', 'B24080_002E', 'B24080_009E', 'B24050_004E', 'B24050_005E', 'B25071_001E', 'B25014_001E', 'B28002_001E', 'B28002_002E', 'B28002_013E', 'B27001_001E', 'B27001_005E']


In [4]:
def get_valid_variables(year, survey_type, target_vars):
    """Checks the Census metadata to see which requested variables actually exist that year."""
    if year == 2020 and survey_type == 'acs1':
        return []

    url = f"https://api.census.gov/data/{year}/acs/{survey_type}/variables.json"

    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            data = response.json()
            valid_vars = [var for var in target_vars if var in data.get('variables', {})]
            return valid_vars
        else:
            print(f"  [!] Warning: Could not fetch dictionary for {year} {survey_type} (Status {response.status_code}). Trying all variables.")
            return target_vars
    except requests.exceptions.RequestException as e:
         print(f"  [!] Warning: Network error fetching dictionary for {year}: {e}. Trying all variables.")
         return target_vars

def fetch_census_data(year, survey_type, original_vars, api_key, max_retries=3):
    print(f"  -> Preparing to fetch {year} {survey_type}...")

    if year == 2020 and survey_type == 'acs1':
        print(f"  [-] Skipping {year} {survey_type} (Standard data not available)")
        return pd.DataFrame()

    # 1. Pre-Check: Find out which variables are safe to request this year
    safe_vars = get_valid_variables(year, survey_type, original_vars)

    if not safe_vars:
        print(f"  [-] None of the requested variables exist in {year} {survey_type}.")
        return pd.DataFrame()

    var_string = ",".join(safe_vars)
    url = f"https://api.census.gov/data/{year}/acs/{survey_type}?get=NAME,{var_string}&for=county:*&in=state:*&key={api_key}"

    # 2. Make the API call with Retries and Timeout
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=30)

            if response.status_code == 200:
                data = response.json()
                headers = data[0]
                rows = data[1:]

                df = pd.DataFrame(rows, columns=headers)
                df['Year'] = year

                # 3. Add back the missing API variables as Nulls so the shape stays consistent
                missing_vars = [var for var in original_vars if var not in df.columns]
                for missing in missing_vars:
                    df[missing] = np.nan

                # 4. Rename the API codes to your human-readable headers
                df.rename(columns=VARIABLE_MAP, inplace=True)

                # 5. Rename the default Census columns for cleanliness
                df.rename(columns={'NAME': 'County_Name', 'state': 'State_FIPS', 'county': 'County_FIPS'}, inplace=True)

                print(f"  [+] Successfully fetched {year} {survey_type} (Dropped/Nulled {len(missing_vars)} variables)")
                return df

            elif response.status_code == 400:
                 print(f"  [x] Failed {year} {survey_type}. Error 400. Even the safe variables were rejected.")
                 return pd.DataFrame()
            else:
                print(f"  [!] Attempt {attempt + 1} failed for {year}. Status Code: {response.status_code}")

        except requests.exceptions.Timeout:
            print(f"  [!] Attempt {attempt + 1} Timed Out for {year}. The Census server is hanging.")
        except requests.exceptions.RequestException as e:
            # This will catch that [Errno 101] Network is unreachable error and retry instead of crashing!
            print(f"  [!] Attempt {attempt + 1} encountered a network error: {e}")

        # Wait 5 seconds before trying the next attempt
        time.sleep(5)

    print(f"  [x] Completely failed to fetch {year} {survey_type} after {max_retries} attempts.")
    return pd.DataFrame()


In [5]:
'''
print("Starting ACS 1-Year Pull...")
acs1_frames = []
for year in years_acs1:
    df_year = fetch_census_data(year, 'acs1', TARGET_VARIABLES, API_KEY)
    if not df_year.empty:
        acs1_frames.append(df_year)
    time.sleep(1)

if acs1_frames:
    final_acs1_df = pd.concat(acs1_frames, ignore_index=True)
    acs1_path = f"{DRIVE_FOLDER}{FILE_PREFIX}acs1_county_2008_2024.csv"
    final_acs1_df.to_csv(acs1_path, index=False)
    print(f"\nSaved ACS 1-Year data to {acs1_path}\n")

'''
print("Starting ACS 5-Year Pull...")
acs5_frames = []
for year in years_acs5:
    df_year = fetch_census_data(year, 'acs5', TARGET_VARIABLES, API_KEY)
    if not df_year.empty:
        acs5_frames.append(df_year)
    time.sleep(1)

if acs5_frames:
    final_acs5_df = pd.concat(acs5_frames, ignore_index=True)
    acs5_path = f"{DRIVE_FOLDER}{FILE_PREFIX}acs5_county_2009_2024.csv"
    final_acs5_df.to_csv(acs5_path, index=False)
    print(f"\nSaved ACS 5-Year data to {acs5_path}")

print("\nData pull complete!")


Starting ACS 5-Year Pull...
  -> Preparing to fetch 2009 acs5...
  [+] Successfully fetched 2009 acs5 (Dropped/Nulled 8 variables)
  -> Preparing to fetch 2010 acs5...
  [+] Successfully fetched 2010 acs5 (Dropped/Nulled 7 variables)
  -> Preparing to fetch 2011 acs5...
  [+] Successfully fetched 2011 acs5 (Dropped/Nulled 7 variables)
  -> Preparing to fetch 2012 acs5...
  [+] Successfully fetched 2012 acs5 (Dropped/Nulled 5 variables)
  -> Preparing to fetch 2013 acs5...
  [+] Successfully fetched 2013 acs5 (Dropped/Nulled 5 variables)
  -> Preparing to fetch 2014 acs5...
  [+] Successfully fetched 2014 acs5 (Dropped/Nulled 5 variables)
  -> Preparing to fetch 2015 acs5...
  [+] Successfully fetched 2015 acs5 (Dropped/Nulled 5 variables)
  -> Preparing to fetch 2016 acs5...
  [+] Successfully fetched 2016 acs5 (Dropped/Nulled 5 variables)
  -> Preparing to fetch 2017 acs5...
  [+] Successfully fetched 2017 acs5 (Dropped/Nulled 2 variables)
  -> Preparing to fetch 2018 acs5...
  [+] Su